In [1]:
from langchain.agents import AgentState

class CallState(AgentState):
    """调用智能体时的智能体状态"""
    model_call_count: int

from langchain.tools import tool,ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_model_call_count(runtime:ToolRuntime):
    """记录模型被调用的次数"""
    count = runtime.state.get("model_call_count",0)
    print("模型被调用次数：",count)

    update = {
        "model_call_count":count+1,
        "messages":[
            ToolMessage(
                content="数据更新成功",
                tool_call_id = runtime.tool_call_id
            )
        ]
    }

    return Command(update=update)


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key = os.getenv("DASHSCOPE_API_KEY"),
    base_url = os.getenv("DASHSCOPE_BASE_URL")
)

agent = create_agent(
    model,
    tools=[update_model_call_count],
    state_schema=CallState,
    checkpointer=InMemorySaver(),
    system_prompt="你在回答用户问题的时候调用工具记录模型被调用次数"
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {"messages":[HumanMessage(content="你好，我是张三")]},
    config
)

for message in response['messages']:
    message.pretty_print()


模型被调用次数： 2
================================ Human Message =================================

你好，我是张三
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_113568bd4821411c9bb5937a)
 Call ID: call_113568bd4821411c9bb5937a
  Args:
================================= Tool Message =================================
Name: update_model_call_count

数据更新成功
================================== Ai Message ==================================

你好，张三！很高兴认识你。有什么我可以帮助你的吗？
================================ Human Message =================================

你好，我是张三
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_ad96f90d711345548d5f3389)
 Call ID: call_ad96f90d711345548d5f3389
  Args:
================================= Tool Message =================================
Name: update_model_call_count

数据更新成功
================================== Ai Message ===================

In [7]:
response = agent.invoke(
    {"messages":[HumanMessage(content="汽车的种类有哪些")]},
    config
)

for message in response['messages']:
    message.pretty_print()


模型被调用次数： 3
================================ Human Message =================================

你好，我是张三
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_113568bd4821411c9bb5937a)
 Call ID: call_113568bd4821411c9bb5937a
  Args:
================================= Tool Message =================================
Name: update_model_call_count

数据更新成功
================================== Ai Message ==================================

你好，张三！很高兴认识你。有什么我可以帮助你的吗？
================================ Human Message =================================

你好，我是张三
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_ad96f90d711345548d5f3389)
 Call ID: call_ad96f90d711345548d5f3389
  Args:
================================= Tool Message =================================
Name: update_model_call_count

数据更新成功
================================== Ai Message ===================